# HypNetPerturb — TargetScores plot notebook

This notebook **only plots results** from files saved by the TargetScores experiments.

It expects an output folder containing (depending on your save format):
- `*.npy` arrays (e.g., `scores_by_mag.npy`, `mag_arr.npy`)
- or `*.npz` bundles
- or `*.pkl` dictionaries

If you have a custom format, adapt the loader cell.

## 0. Imports

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import pickle

## 1. Load results from disk

In [ ]:
# Point this to your experiment output directory
OUTDIR = Path("outputs")   # change if needed

# --- Loader helpers ---
def load_pickle(path):
    with open(path, "rb") as f:
        return pickle.load(f)

def try_load():
    # Try common filenames (edit to match your saving conventions)
    candidates = {
        "npz": OUTDIR / "TargetScores_results.npz",
        "scores_npy": OUTDIR / "scores_by_mag.npy",
        "mag_npy": OUTDIR / "mag_arr.npy",
        "pkl": OUTDIR / "TargetScores_by_mag.pkl",
    }

    if candidates["npz"].exists():
        data = np.load(candidates["npz"], allow_pickle=True)
        return dict(data)

    if candidates["pkl"].exists():
        d = load_pickle(candidates["pkl"])
        return {"TargetScores_by_mag": d}

    if candidates["scores_npy"].exists():
        scores = np.load(candidates["scores_npy"], allow_pickle=True)
        mag = np.load(candidates["mag_npy"], allow_pickle=True) if candidates["mag_npy"].exists() else None
        return {"scores": scores, "mag": mag}

    raise FileNotFoundError(
        "Could not find expected result files in OUTDIR. "
        "Update OUTDIR and/or candidate filenames in this cell."
    )

data = try_load()
data.keys()

## 2. Standardize to `(S, mag_sorted)`

In [ ]:
def standardize(data):
    # dict-of-mags style
    if "TargetScores_by_mag" in data and isinstance(data["TargetScores_by_mag"], dict):
        d = data["TargetScores_by_mag"]
        mag_sorted = np.array(sorted(d.keys()))
        S = np.vstack([np.asarray(d[m]).ravel() for m in mag_sorted])
        return S, mag_sorted

    # npz style
    if "S" in data and "mag" in data:
        S = np.asarray(data["S"])
        mag = np.asarray(data["mag"])
        return S, mag

    # raw arrays
    if "scores" in data:
        S = np.asarray(data["scores"])
        mag = np.asarray(data["mag"]) if data.get("mag", None) is not None else np.arange(S.shape[0])
        return S, mag

    # attempt to detect any 2D array + mag vector
    for k, v in data.items():
        if isinstance(v, np.ndarray) and v.ndim == 2:
            S = v
            mag = data.get("mag_arr", np.arange(S.shape[0]))
            return S, np.asarray(mag)

    raise ValueError("Could not standardize loaded data. Inspect `data` and edit `standardize()`.")

S, mag_sorted = standardize(data)
S.shape, mag_sorted[:5]

## 3. Plots

In [ ]:
mean_over_nodes = np.nanmean(S, axis=1)
std_over_nodes  = np.nanstd(S, axis=1)

plt.figure(figsize=(7,4))
plt.errorbar(mag_sorted, mean_over_nodes, yerr=std_over_nodes, fmt='-o', capsize=3)
plt.xlabel("Perturbation magnitude")
plt.ylabel("kNHD score (mean ± std across nodes)")
plt.title("Global sensitivity vs magnitude")
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(10,5))
plt.imshow(S, aspect='auto', interpolation='nearest')
plt.colorbar(label="kNHD score")
plt.yticks(ticks=np.arange(len(mag_sorted)), labels=[str(m) for m in mag_sorted])
plt.xlabel("Node index")
plt.ylabel("Magnitude")
plt.title("Node-wise kNHD scores across magnitudes")
plt.tight_layout()
plt.show()

## Appendix: Original plotting script (reference)

In [ ]:
# Reference: plot_TargetScoresXmag.py

#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
Created on Wed Nov 15 09:53:50 2023

@author: alice.longhena
"""

#%%----------- INIT

import pickle
import numpy as np
import matplotlib.pyplot as plt
import scipy.stats as st 
from LIBS.utils import *

colors = ['#e37c1d', '#0076c0', '#f1b682', '#bacfec', 'tab:blue', 'tab:orange', 'tab:green', 'tab:red', 'tab:purple', 'tab:brown', 'tab:pink', 'tab:gray', 'tab:olive', 'tab:cyan']

n_G = 3

mag_arr = np.array([20/100, 40/100, 60/100, 80/100])
alpha_dict_WS = {mag_arr[0]:1/1000, mag_arr[1]:1/20000, mag_arr[2]:1/20000, mag_arr[3]:1/1000}
alpha_dict_ER = {mag_arr[0]:1/100, mag_arr[1]:1/600, mag_arr[2]:1/600, mag_arr[3]:1/100}
alpha_dict_BA = {mag_arr[0]:1/100, mag_arr[1]:1/1000, mag_arr[2]:1/1000, mag_arr[3]:1/100}
alpha_dict_HI = {mag_arr[0]:1/100, mag_arr[1]:1/1200, mag_arr[2]:1/1200, mag_arr[3]:1/100}


# model: hierarchical/WattsStrogatz/ErdosRenyi/BarabasiAlbert
model = 'real-epi'

# HI/WS/ER/BA
alpha_dict = assign_alpha_dict(model,mag_arr)

N = 150 # 150 except for HI 128

k = 10

output_folder = '/Users/alice.longhena/Desktop/GitHub/kNHD-score-PerturbationDetection/OUTPUT/output_3G_iso'
media_folder = '/Users/alice.longhena/Desktop/GitHub/kNHD-score-PerturbationDetection/MEDIA/media_3G_iso'


#%% ------------ DEGREE Class plot

fig =plt.figure(figsize = (40,11))
plt.rcParams['axes.linewidth']=2.0
fig.suptitle('%s graph model N=%i'%(model,N), fontsize=30, y=1.0005)

for i,m in enumerate(mag_arr):

    alpha = alpha_dict[m]
    
    ## LOAD dictionaries
    degreeXscore_global = pickle.load(open(output_folder+'/degreeXscore_global_%s_dis-mag%.2f_nG%i.pkl'%(model,m,n_G), 'rb'))
    degreeXscore_target = pickle.load(open(output_folder+'/degreeXscore_target_%s_dis-mag%.2f_nG%i.pkl'%(model,m,n_G), 'rb'))
    degreeXscore_target2 = pickle.load(open(output_folder+'/degreeXscore_target2_%s_dis-mag%.2f_nG%i.pkl'%(model,m,n_G), 'rb'))
    
    # Creation of the axis
    ax = fig.add_subplot(1,4,i+1)
    
    # the x
    degree_order = np.argsort(list(degreeXscore_global))
    degrees = np.sort(list(degreeXscore_global))
    ld = len(degrees)
    
    # y and dy
    ref = list(degreeXscore_global.values())
    ref_avg = [np.mean(ref[i]) for i in range(ld)]
    ref_avg_avg = np.mean(ref_avg)
    
    target_Score_values = list(degreeXscore_target.values())
    target_Score_avg = np.array([np.mean(target_Score_values[i]-ref[i]) for i in range(ld)])[degree_order] # target score - ref
    target_Score_ci = np.array([st.t.interval(confidence=0.95, df=len(target_Score_values[i]-ref[i])-1, 
                                        loc=np.mean(target_Score_values[i]-ref[i]), 
                                        scale=st.sem(target_Score_values[i]-ref[i])) for i in range(ld)])[degree_order]
    
    ax.plot(degrees, target_Score_avg,'-', linewidth=2.5, markersize=8, alpha=1, color='tab:orange', markeredgecolor=colors[0], markerfacecolor='white', markeredgewidth=2, label='Target %iNHypD score mean - BL'%k)
    ax.fill_between(degrees, target_Score_ci[:,0], target_Score_ci[:,1], alpha=0.4, color='tab:orange')
    
    target2_Score_values = list(degreeXscore_target2.values())
    target2_Score_avg = np.array([np.mean(target2_Score_values[i]-ref[i]) for i in range(ld)])[degree_order] # target score - ref
    target2_Score_ci = np.array([st.t.interval(confidence=0.95, df=len(target2_Score_values[i]-ref[i])-1, 
                                        loc=np.mean(target2_Score_values[i]-ref[i]), 
                                        scale=st.sem(target2_Score_values[i]-ref[i])) for i in range(ld)])[degree_order]

    
    ax.plot(degrees, target2_Score_avg,'-', linewidth=2.5, markersize=8, alpha=1, color='tab:green', markeredgecolor=colors[1], markerfacecolor='white', markeredgewidth=2, label='Target NNavg %iNHypD score mean - BL'%k)
    ax.fill_between(degrees, target2_Score_ci[:,0], target2_Score_ci[:,1], alpha=0.4, color='tab:green')

    ax.axhline(y=0, linestyle='--', linewidth='3', color=colors[11], label='BL(baseline): All nodes %iNHypD score avg'%k)
    
    # Parameters
    
    ax.set_xlabel(r'Perturbed node degree', style = 'italic', fontsize = 30, labelpad=20)
   
    if i==0:
        ax.set_ylabel(r'Score - BL', style = 'italic', fontsize = 30)
        ax.legend(fontsize = 23, ncol=1, loc='upper center', fancybox=True, framealpha=0.4, mode='expand')
   
    ax.tick_params(axis = 'x', which = 'both',labelsize = 30, width = 2, length = 7 ) #ax1.tick_params(axis='y', labelcolor=color)
    ax.tick_params(axis = 'y', which = 'both' ,labelsize = 30, width = 2, length = 7 )
   
    ax.set_title('Perturbation ' + str(int(m*100)) + '%' +  r'  $\overline{BL}$ = %.2f'%(ref_avg_avg) , fontsize = 30, pad=20)
    #ax.set_xscale('log')

   
fig.tight_layout()
   
plt.savefig(media_folder+'/degreeXscoresDiff_allMags_%s_dis_nG%i.pdf'%(model,n_G))
#plt.savefig(media_folder+'/clusteringXscoresDiff_Mags%.2fto%.2f_%s_dis_nG%i.pdf'%(mag_arr[0], mag_arr[3],model,n_G))
plt.show()    
       


#%%------------ CLUSTERING Class plot

fig =plt.figure(figsize = (40,11))
plt.rcParams['axes.linewidth']=2.0
fig.suptitle('%s graph model N=%i'%(model,N), fontsize=30, y=1.0005)

for i,m in enumerate(mag_arr): 

    alpha = alpha_dict[m]
    
    ## LOAD dictionaries
    clusteringXscore_global = pickle.load(open(output_folder+'/clusteringXscore_global_%s_dis-mag%.2f_a%.3f_nG%i.pkl'%(model,m,alpha,n_G), 'rb'))
    clusteringXscore_target = pickle.load(open(output_folder+'/clusteringXscore_target_%s_dis-mag%.2f_a%.3f_nG%i.pkl'%(model,m,alpha,n_G), 'rb'))
    clusteringXscore_target2 = pickle.load(open(output_folder+'/clusteringXscore_target2_%s_dis-mag%.2f_a%.3f_nG%i.pkl'%(model,m,alpha,n_G), 'rb'))
    
    # Creation of the axis
    ax = fig.add_subplot(1,4,i+1)
    
    # the x
    clustering_order = np.argsort(list(clusteringXscore_global))
    clusterings = np.sort(list(clusteringXscore_global))/100 #for clustering_int2
    ld = len(clusterings)
    print(clusterings)

    # y and dy
    ref = list(clusteringXscore_global.values())
    ref_avg = [np.mean(ref[i]) for i in range(ld)]
    ref_avg_avg = np.mean(ref_avg)
    
    target_Score_values = list(clusteringXscore_target.values())
    target_Score_avg = np.array([np.mean(target_Score_values[i]-ref[i]) for i in range(ld)])[clustering_order] # target score - ref
    target_Score_ci = np.array([st.t.interval(confidence=0.95, df=len(target_Score_values[i]-ref[i])-1, 
                                        loc=np.mean(target_Score_values[i]-ref[i]), 
                                        scale=st.sem(target_Score_values[i]-ref[i])) for i in range(ld)])[clustering_order]
    
    ax.plot(clusterings, target_Score_avg,'-', linewidth=2.5, markersize=8, alpha=1, color='tab:orange', markeredgecolor=colors[0], markerfacecolor='white', markeredgewidth=2, label='Target %iNHypD score mean - BL'%k)
    ax.fill_between(clusterings, target_Score_ci[:,0], target_Score_ci[:,1], alpha=0.4, color='tab:orange')
    
    target2_Score_values = list(clusteringXscore_target2.values())
    target2_Score_avg = np.array([np.mean(target2_Score_values[i]-ref[i]) for i in range(ld)])[clustering_order] # target score - ref
    target2_Score_ci = np.array([st.t.interval(confidence=0.95, df=len(target2_Score_values[i]-ref[i])-1, 
                                        loc=np.mean(target2_Score_values[i]-ref[i]), 
                                        scale=st.sem(target2_Score_values[i]-ref[i])) for i in range(ld)])[clustering_order]

    
    ax.plot(clusterings, target2_Score_avg,'-', linewidth=2.5, markersize=8, alpha=1, color='tab:green', markeredgecolor=colors[1], markerfacecolor='white', markeredgewidth=2, label='Target NNavg %iNHypD score mean - BL'%k)
    ax.fill_between(clusterings, target2_Score_ci[:,0], target2_Score_ci[:,1], alpha=0.4, color='tab:green')

    ax.axhline(y=0, linestyle='--', linewidth='3', color=colors[11], label='BL(baseline): All nodes %iNHypD score avg'%k)
    
    # Parameters
    
    ax.set_xlabel(r'Perturbed node clustering', style = 'italic', fontsize = 30, labelpad=20)
   
    if i==0:
        ax.set_ylabel(r'Score - BL', style = 'italic', fontsize = 30)
        ax.legend(fontsize = 23, ncol=1, loc='upper center', fancybox=True, framealpha=0.4, mode='expand')
   
    ax.tick_params(axis = 'x', which = 'both',labelsize = 30, width = 2, length = 7 ) #ax1.tick_params(axis='y', labelcolor=color)
    ax.tick_params(axis = 'y', which = 'both' ,labelsize = 30, width = 2, length = 7 )
   
    ax.set_title('Perturbation ' + str(int(m*100)) + '%' +  r'  $\overline{BL}$ = %.2f'%(ref_avg_avg) , fontsize = 30, pad=20)
    #ax.set_xscale('log')

   
fig.tight_layout()
   
plt.savefig(media_folder+'/clusteringXscoresDiff_allMags_%s_dis_nG%i.pdf'%(model,n_G))
#plt.savefig(media_folder+'/clusteringXscoresDiff_Mags%.2fto%.2f_%s_dis_nG%i.pdf'%(mag_arr[0], mag_arr[3],model,n_G))
plt.show()    


"""

## Construct DATAFRAMES
degrees_perturbed = np.array(list(degreeXscore_global.keys()))

global_avg_avg = 0
degrees_arr = np.array([])
scores_arr = np.array([])
scores_hue_arr = np.array([])

for d in degrees_perturbed:
    
    l = len(degreeXscore_global[d])
    degrees_arr = np.append(degrees_arr,np.ones(l*2)*d)

    scores_arr = np.append(scores_arr,degreeXscore_target[d]-degreeXscore_global[d])
    scores_hue_arr = np.append(scores_hue_arr,np.array(['target']*l))

    scores_arr = np.append(scores_arr,degreeXscore_target2[d]-degreeXscore_global[d])
    scores_hue_arr = np.append(scores_hue_arr,np.array(['target 2 avg']*l))  
    
    global_avg_avg += np.average(degreeXscore_global[d])

global_avg_avg = global_avg_avg/len(degrees_perturbed)

df_degree_score = pd.DataFrame({'degrees':degrees_arr, 'score at target - global avg score':scores_arr, 'score cat':scores_hue_arr})

s = sns.lineplot(x="degrees", y="score at target - global avg score", hue="score cat", err_style="band",
             errorbar ='ci',
             estimator=np.mean,
             data=df_degree_score, 
             ax = axes[0])

axes[0].set_title("mag %.2f - global avg estimate = %.2f"%(m,global_avg_avg), fontsize=20)
axes[0].axhline(y=0, color="black", linestyle="--", label='global avg')
#axes[0].text(x=20, y=0+0.5, s='global avg score estimate = %.2f'%global_avg_avg, ha='left', fontsize=9, alpha=0.7)
axes[0].legend()
sns.move_legend(s, loc = 'upper left', frameon=True, fontsize=15)


### AX 1
   
## LOAD dictionaries
m = mag_arr[1] 
alpha = alpha_dict[m]
degreeXscore_global = pickle.load(open(output_folder+'/degreeXscore_global_%s_dis-mag%.2f_a%.3f_nG%i.pkl'%(model,m,alpha,n_G), 'rb'))
degreeXscore_target = pickle.load(open(output_folder+'/degreeXscore_target_%s_dis-mag%.2f_a%.3f_nG%i.pkl'%(model,m,alpha,n_G), 'rb'))
degreeXscore_target2 = pickle.load(open(output_folder+'/degreeXscore_target2_%s_dis-mag%.2f_a%.3f_nG%i.pkl'%(model,m,alpha,n_G), 'rb'))

## Construct DATAFRAMES
degrees_perturbed = np.array(list(degreeXscore_global.keys()))

global_avg_avg = 0
degrees_arr = np.array([])
scores_arr = np.array([])
scores_hue_arr = np.array([])

for d in degrees_perturbed:
    
    l = len(degreeXscore_global[d])
    degrees_arr = np.append(degrees_arr,np.ones(l*2)*d)

    scores_arr = np.append(scores_arr,degreeXscore_target[d]-degreeXscore_global[d])
    scores_hue_arr = np.append(scores_hue_arr,np.array(['target']*l))

    scores_arr = np.append(scores_arr,degreeXscore_target2[d]-degreeXscore_global[d])
    scores_hue_arr = np.append(scores_hue_arr,np.array(['target 2 avg']*l))  
    
    global_avg_avg += np.average(degreeXscore_global[d])

global_avg_avg = global_avg_avg/len(degrees_perturbed)

df_degree_score = pd.DataFrame({'degrees':degrees_arr, 'score at target - global avg score':scores_arr, 'score cat':scores_hue_arr})

s = sns.lineplot(x="degrees", y="score at target - global avg score", hue="score cat", err_style="band",
             errorbar ='ci',
             estimator=np.mean,
             data=df_degree_score, 
             ax = axes[1])

axes[1].set_title("mag %.2f - global avg estimate = %.2f"%(m,global_avg_avg), fontsize=20)
axes[1].axhline(y=0, color="black", linestyle="--", label='global avg')
#axes[0].text(x=20, y=0+0.5, s='global avg score estimate = %.2f'%global_avg_avg, ha='left', fontsize=9, alpha=0.7)
axes[1].legend()
sns.move_legend(s, loc='upper left', frameon=True, fontsize=15)



### AX 2
   
## LOAD dictionaries
m = mag_arr[2] 
alpha = alpha_dict[m]
degreeXscore_global = pickle.load(open(output_folder+'/degreeXscore_global_%s_dis-mag%.2f_a%.3f_nG%i.pkl'%(model,m,alpha,n_G), 'rb'))
degreeXscore_target = pickle.load(open(output_folder+'/degreeXscore_target_%s_dis-mag%.2f_a%.3f_nG%i.pkl'%(model,m,alpha,n_G), 'rb'))
degreeXscore_target2 = pickle.load(open(output_folder+'/degreeXscore_target2_%s_dis-mag%.2f_a%.3f_nG%i.pkl'%(model,m,alpha,n_G), 'rb'))

## Construct DATAFRAMES
degrees_perturbed = np.array(list(degreeXscore_global.keys()))

global_avg_avg = 0
degrees_arr = np.array([])
scores_arr = np.array([])
scores_hue_arr = np.array([])

for d in degrees_perturbed:
    
    l = len(degreeXscore_global[d])
    degrees_arr = np.append(degrees_arr,np.ones(l*2)*d)

    scores_arr = np.append(scores_arr,degreeXscore_target[d]-degreeXscore_global[d])
    scores_hue_arr = np.append(scores_hue_arr,np.array(['target']*l))

    scores_arr = np.append(scores_arr,degreeXscore_target2[d]-degreeXscore_global[d])
    scores_hue_arr = np.append(scores_hue_arr,np.array(['target 2 avg']*l))  
    
    global_avg_avg += np.average(degreeXscore_global[d])

global_avg_avg = global_avg_avg/len(degrees_perturbed)

df_degree_score = pd.DataFrame({'degrees':degrees_arr, 'score at target - global avg score':scores_arr, 'score cat':scores_hue_arr})

s = sns.lineplot(x="degrees", y="score at target - global avg score", hue="score cat", err_style="band",
             errorbar ='ci',
             estimator=np.mean,
             data=df_degree_score, 
             ax = axes[2])

axes[2].set_title("mag %.2f - global avg estimate = %.2f"%(m,global_avg_avg), fontsize=20)
axes[2].axhline(y=0, color="black", linestyle="--", label='global avg')
#axes[0].text(x=20, y=0+0.5, s='global avg score estimate = %.2f'%global_avg_avg, ha='left', fontsize=9, alpha=0.7)
axes[2].legend()
sns.move_legend(s, loc='upper left', frameon=True, fontsize=15)



### AX 3
   
## LOAD dictionaries
m = mag_arr[3] 
alpha = alpha_dict[m]
degreeXscore_global = pickle.load(open(output_folder+'/degreeXscore_global_%s_dis-mag%.2f_a%.3f_nG%i.pkl'%(model,m,alpha,n_G), 'rb'))
degreeXscore_target = pickle.load(open(output_folder+'/degreeXscore_target_%s_dis-mag%.2f_a%.3f_nG%i.pkl'%(model,m,alpha,n_G), 'rb'))
degreeXscore_target2 = pickle.load(open(output_folder+'/degreeXscore_target2_%s_dis-mag%.2f_a%.3f_nG%i.pkl'%(model,m,alpha,n_G), 'rb'))

## Construct DATAFRAMES
degrees_perturbed = np.array(list(degreeXscore_global.keys()))

global_avg_avg = 0
degrees_arr = np.array([])
scores_arr = np.array([])
scores_hue_arr = np.array([])

for d in degrees_perturbed:
    
    l = len(degreeXscore_global[d])
    degrees_arr = np.append(degrees_arr,np.ones(l*2)*d)

    scores_arr = np.append(scores_arr,degreeXscore_target[d]-degreeXscore_global[d])
    scores_hue_arr = np.append(scores_hue_arr,np.array(['target']*l))

    scores_arr = np.append(scores_arr,degreeXscore_target2[d]-degreeXscore_global[d])
    scores_hue_arr = np.append(scores_hue_arr,np.array(['target 2 avg']*l))  
    
    global_avg_avg += np.average(degreeXscore_global[d])

global_avg_avg = global_avg_avg/len(degrees_perturbed)

df_degree_score = pd.DataFrame({'degrees':degrees_arr, 'score at target - global avg score':scores_arr, 'score cat':scores_hue_arr})

s = sns.lineplot(x="degrees", y="score at target - global avg score", hue="score cat", err_style="band",
             errorbar ='ci',
             estimator=np.mean,
             data=df_degree_score, 
             ax = axes[3])

axes[3].set_title("mag %.2f - global avg estimate = %.2f"%(m,global_avg_avg), fontsize=20)
axes[3].axhline(y=0, color="black", linestyle="--", label='global avg')
#axes[0].text(x=20, y=0+0.5, s='global avg score estimate = %.2f'%global_avg_avg, ha='left', fontsize=9, alpha=0.7)
axes[3].legend()
sns.move_legend(s, loc='upper left', frameon=True, fontsize=15)


plt.savefig(media_folder+"/degreesXscoresDiff_allMags_%s_dis_nG%i.pdf"%(model,n_G))
plt.show()



#--- CLUSTERING

fig, axes = plt.subplots(1, 4, figsize=(45, 10))
fig.suptitle('difference from global average score - %s N%i'%(model,N),fontsize=20, y=0.98)

### AX 0
   
## LOAD dictionaries
m = mag_arr[0] 
alpha = alpha_dict[m]
clusteringXscore_global = pickle.load(open(output_folder+'/clusteringXscore_global_%s_dis-mag%.2f_a%.3f_nG%i.pkl'%(model,m,alpha,n_G), 'rb'))
clusteringXscore_target = pickle.load(open(output_folder+'/clusteringXscore_target_%s_dis-mag%.2f_a%.3f_nG%i.pkl'%(model,m,alpha,n_G), 'rb'))
clusteringXscore_target2 = pickle.load(open(output_folder+'/clusteringXscore_target2_%s_dis-mag%.2f_a%.3f_nG%i.pkl'%(model,m,alpha,n_G), 'rb'))

## Construct DATAFRAMES
clusterings_perturbed = np.array(list(clusteringXscore_global.keys()))

global_avg_avg = 0
clusterings_arr = np.array([])
scores_arr = np.array([])
scores_hue_arr = np.array([])

for d in clusterings_perturbed:
    
    l = len(clusteringXscore_global[d])
    clusterings_arr = np.append(clusterings_arr,np.ones(l*2)*d)

    scores_arr = np.append(scores_arr,clusteringXscore_target[d]-clusteringXscore_global[d])
    scores_hue_arr = np.append(scores_hue_arr,np.array(['target']*l))

    scores_arr = np.append(scores_arr,clusteringXscore_target2[d]-clusteringXscore_global[d])
    scores_hue_arr = np.append(scores_hue_arr,np.array(['target 2 avg']*l))  
    
    global_avg_avg += np.average(clusteringXscore_global[d])

global_avg_avg = global_avg_avg/len(clusterings_perturbed)

df_clustering_score = pd.DataFrame({'clusterings':clusterings_arr, 'score at target - global avg score':scores_arr, 'score cat':scores_hue_arr})

s = sns.lineplot(x="clusterings", y="score at target - global avg score", hue="score cat", err_style="band",
             errorbar ='ci',
             estimator=np.mean,
             data=df_clustering_score, 
             ax = axes[0])

axes[0].set_title("mag %.2f - global avg estimate = %.2f"%(m,global_avg_avg), fontsize=20)
axes[0].axhline(y=0, color="black", linestyle="--", label='global avg')
#axes[0].text(x=20, y=0+0.5, s='global avg score estimate = %.2f'%global_avg_avg, ha='left', fontsize=9, alpha=0.7)
axes[0].legend()
sns.move_legend(s, loc = 'upper left', frameon=True, fontsize=15)


### AX 1
   
## LOAD dictionaries
m = mag_arr[1] 
alpha = alpha_dict[m]
clusteringXscore_global = pickle.load(open(output_folder+'/clusteringXscore_global_%s_dis-mag%.2f_a%.3f_nG%i.pkl'%(model,m,alpha,n_G), 'rb'))
clusteringXscore_target = pickle.load(open(output_folder+'/clusteringXscore_target_%s_dis-mag%.2f_a%.3f_nG%i.pkl'%(model,m,alpha,n_G), 'rb'))
clusteringXscore_target2 = pickle.load(open(output_folder+'/clusteringXscore_target2_%s_dis-mag%.2f_a%.3f_nG%i.pkl'%(model,m,alpha,n_G), 'rb'))

## Construct DATAFRAMES
clusterings_perturbed = np.array(list(clusteringXscore_global.keys()))

global_avg_avg = 0
clusterings_arr = np.array([])
scores_arr = np.array([])
scores_hue_arr = np.array([])

for d in clusterings_perturbed:
    
    l = len(clusteringXscore_global[d])
    clusterings_arr = np.append(clusterings_arr,np.ones(l*2)*d)

    scores_arr = np.append(scores_arr,clusteringXscore_target[d]-clusteringXscore_global[d])
    scores_hue_arr = np.append(scores_hue_arr,np.array(['target']*l))

    scores_arr = np.append(scores_arr,clusteringXscore_target2[d]-clusteringXscore_global[d])
    scores_hue_arr = np.append(scores_hue_arr,np.array(['target 2 avg']*l))  
    
    global_avg_avg += np.average(clusteringXscore_global[d])

global_avg_avg = global_avg_avg/len(clusterings_perturbed)

df_clustering_score = pd.DataFrame({'clusterings':clusterings_arr, 'score at target - global avg score':scores_arr, 'score cat':scores_hue_arr})

s = sns.lineplot(x="clusterings", y="score at target - global avg score", hue="score cat", err_style="band",
             errorbar ='ci',
             estimator=np.mean,
             data=df_clustering_score, 
             ax = axes[1])

axes[1].set_title("mag %.2f - global avg estimate = %.2f"%(m,global_avg_avg), fontsize=20)
axes[1].axhline(y=0, color="black", linestyle="--", label='global avg')
#axes[0].text(x=20, y=0+0.5, s='global avg score estimate = %.2f'%global_avg_avg, ha='left', fontsize=9, alpha=0.7)
axes[1].legend()
sns.move_legend(s, loc = 'upper left', frameon=True, fontsize=15)


### AX 2
   
## LOAD dictionaries
m = mag_arr[2] 
alpha = alpha_dict[m]
clusteringXscore_global = pickle.load(open(output_folder+'/clusteringXscore_global_%s_dis-mag%.2f_a%.3f_nG%i.pkl'%(model,m,alpha,n_G), 'rb'))
clusteringXscore_target = pickle.load(open(output_folder+'/clusteringXscore_target_%s_dis-mag%.2f_a%.3f_nG%i.pkl'%(model,m,alpha,n_G), 'rb'))
clusteringXscore_target2 = pickle.load(open(output_folder+'/clusteringXscore_target2_%s_dis-mag%.2f_a%.3f_nG%i.pkl'%(model,m,alpha,n_G), 'rb'))

## Construct DATAFRAMES
clusterings_perturbed = np.array(list(clusteringXscore_global.keys()))

global_avg_avg = 0
clusterings_arr = np.array([])
scores_arr = np.array([])
scores_hue_arr = np.array([])

for d in clusterings_perturbed:
    
    l = len(clusteringXscore_global[d])
    clusterings_arr = np.append(clusterings_arr,np.ones(l*2)*d)

    scores_arr = np.append(scores_arr,clusteringXscore_target[d]-clusteringXscore_global[d])
    scores_hue_arr = np.append(scores_hue_arr,np.array(['target']*l))

    scores_arr = np.append(scores_arr,clusteringXscore_target2[d]-clusteringXscore_global[d])
    scores_hue_arr = np.append(scores_hue_arr,np.array(['target 2 avg']*l))  
    
    global_avg_avg += np.average(clusteringXscore_global[d])

global_avg_avg = global_avg_avg/len(clusterings_perturbed)

df_clustering_score = pd.DataFrame({'clusterings':clusterings_arr, 'score at target - global avg score':scores_arr, 'score cat':scores_hue_arr})

s = sns.lineplot(x="clusterings", y="score at target - global avg score", hue="score cat", err_style="band",
             errorbar ='ci',
             estimator=np.mean,
             data=df_clustering_score, 
             ax = axes[2])

axes[2].set_title("mag %.2f - global avg estimate = %.2f"%(m,global_avg_avg), fontsize=20)
axes[2].axhline(y=0, color="black", linestyle="--", label='global avg')
#axes[0].text(x=20, y=0+0.5, s='global avg score estimate = %.2f'%global_avg_avg, ha='left', fontsize=9, alpha=0.7)
axes[2].legend()
sns.move_legend(s, loc = 'upper left', frameon=True, fontsize=15)


### AX 3
   
## LOAD dictionaries
m = mag_arr[3] 
alpha = alpha_dict[m]
clusteringXscore_global = pickle.load(open(output_folder+'/clusteringXscore_global_%s_dis-mag%.2f_a%.3f_nG%i.pkl'%(model,m,alpha,n_G), 'rb'))
clusteringXscore_target = pickle.load(open(output_folder+'/clusteringXscore_target_%s_dis-mag%.2f_a%.3f_nG%i.pkl'%(model,m,alpha,n_G), 'rb'))
clusteringXscore_target2 = pickle.load(open(output_folder+'/clusteringXscore_target2_%s_dis-mag%.2f_a%.3f_nG%i.pkl'%(model,m,alpha,n_G), 'rb'))

## Construct DATAFRAMES
clusterings_perturbed = np.array(list(clusteringXscore_global.keys()))

global_avg_avg = 0
clusterings_arr = np.array([])
scores_arr = np.array([])
scores_hue_arr = np.array([])

for d in clusterings_perturbed:
    
    l = len(clusteringXscore_global[d])
    clusterings_arr = np.append(clusterings_arr,np.ones(l*2)*d)

    scores_arr = np.append(scores_arr,clusteringXscore_target[d]-clusteringXscore_global[d])
    scores_hue_arr = np.append(scores_hue_arr,np.array(['target']*l))

    scores_arr = np.append(scores_arr,clusteringXscore_target2[d]-clusteringXscore_global[d])
    scores_hue_arr = np.append(scores_hue_arr,np.array(['target 2 avg']*l))  
    
    global_avg_avg += np.average(clusteringXscore_global[d])

global_avg_avg = global_avg_avg/len(clusterings_perturbed)

df_clustering_score = pd.DataFrame({'clusterings':clusterings_arr, 'score at target - global avg score':scores_arr, 'score cat':scores_hue_arr})

s = sns.lineplot(x="clusterings", y="score at target - global avg score", hue="score cat", err_style="band",
             errorbar ='ci',
             estimator=np.mean,
             data=df_clustering_score, 
             ax = axes[3])

axes[3].set_title("mag %.2f - global avg estimate = %.2f"%(m,global_avg_avg), fontsize=20)
axes[3].axhline(y=0, color="black", linestyle="--", label='global avg')
#axes[0].text(x=20, y=0+0.5, s='global avg estimate = %.2f'%global_avg_avg, ha='left', fontsize=9, alpha=0.7)
axes[3].legend()
sns.move_legend(s, loc = 'upper left', frameon=True, fontsize=15)


plt.savefig(media_folder+"/clusteringsXscoresDiff_allMags_%s_dis_nG%i.pdf"%(model,n_G))
plt.show()
"""
# %%
